# The feature catalog (Prompt #8, Steps 2-13, 53)

One registry for every family - returns, volatility, autocorrelation, regression, OU, FFT, wavelet, regime, microstructure, time, interactions - with windows, warm-up, cost and a live-safe flag. Non-causal constructions are catalogued only to record why they are excluded; they are never computed.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.features` / `xauusd_quant.alpha` / `xauusd_quant.research.feature_*` and read back from `results/feature_research/`. Generate those first:

```
xq build-feature-factory --timeframe 1h      # versioned causal matrix + targets
xq feature-research --timeframe 1h           # every stage, statuses, ledger, figures
xq feature-research --all                    # 5m, 15m, 30m, 1h (one child process each)
xq feature-research --compare                # cross-timeframe tables
```

**Descriptive research only.** Statuses (`strong_candidate` ... `failed_null`) are statistical information about rank ICs against circular-shift, pipeline and noise-feature nulls - not signals, not a model, no cost, no PnL. Features at `t` use bars `<= t`; targets live in their own namespace (`target_*`) and are never joined into the matrix.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.features.factory import current_version_dir
from xauusd_quant.features.factory_config import load_features_config

fcfg = load_features_config(ROOT / "config" / "features.yaml")

TIMEFRAME = "1h"       # 5m | 15m | 30m | 1h - change and re-run

base = fcfg.results_path / TIMEFRAME
if not base.exists():
    print(f"No research for {TIMEFRAME}. Run:  xq feature-research --timeframe {TIMEFRAME}")

def table(name):
    """Load a previously generated research table (None if it was not written)."""
    path = base / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name):
    path = base / "figures" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"(no figure {path.name})")

def load_json(name):
    path = base / name
    return json.loads(path.read_text()) if path.exists() else {}

summary = load_json("summary.json")
{k: v for k, v in summary.items() if not isinstance(v, (dict, list))}

## Registry of the stored matrix

In [ ]:
mdir = current_version_dir(fcfg.factory_path, TIMEFRAME)
registry = pl.DataFrame(json.loads((mdir / 'registry.json').read_text()), infer_schema_length=None)
registry.select('name', 'family', 'window', 'min_history', 'cost', 'live_safe', 'prior_status', 'invalid_reason')

In [ ]:
registry.group_by('family').len().sort('family')

## Catalogued but excluded (non-causal constructions)

In [ ]:
catalog = pl.DataFrame(json.loads((mdir / 'feature_catalog.json').read_text()), infer_schema_length=None)
catalog.filter(pl.col('family') == 'non_causal')

## Integrity gate and dataset lineage

In [ ]:
load_json('integrity_gate.json')

## Missingness, distributions and drift (feature values only)

In [ ]:
table('quality/missingness').sort('missing_share', descending=True).head(20)

In [ ]:
table('quality/feature_drift_summary')

## Live cost per family

In [ ]:
table('cost/cost_profile')

In [ ]:
table('cost/live_buffer_requirements')

## Family coverage by status

In [ ]:
plot('family_coverage')